In [ ]:
import os
import json
import random
import numpy as np
from pydub import AudioSegment
from tqdm import tqdm

def generate_mixes(add_noise_flag=False):
    # Конфигурация
    config = {
        "audio_dir": os.path.abspath("common_voice_processed/audio/concatenated"),
        "meta_dir": os.path.abspath("common_voice_processed/metadata/concatenated"),
        "noise_dir": os.path.abspath("audio_noise"),
        "output_dir": os.path.abspath("common_voice_processed/audio/mixed"),
        "output_meta_dir": os.path.abspath("common_voice_processed/metadata/mixed"),
        "patterns": ["Частичное перекрытие", "Параллельный диалог"],
        "duration_ranges": [(5,15), (15,30), (30,60)],
        "snr_levels": [-5, 0, 5],
        "test_mode": True,
        "add_noise": add_noise_flag
    }

    # Загрузка данных
    valid_records = []
    for meta_file in os.listdir(config["meta_dir"]):
        if not meta_file.endswith('.json'):
            continue
            
        meta_path = os.path.join(config["meta_dir"], meta_file)
        audio_file = meta_file.replace('.json', '.wav')
        audio_path = os.path.join(config["audio_dir"], audio_file)
        
        if not os.path.exists(audio_path):
            continue
            
        try:
            with open(meta_path, 'r', encoding='utf-8') as f:
                meta = json.load(f)
                if 'speaker_id' not in meta or 'total_duration' not in meta:
                    continue
                    
                valid_records.append({
                    "speaker_id": meta["speaker_id"],
                    "duration": meta["total_duration"],
                    "audio_path": audio_path,
                    "meta": meta
                })
        except Exception as e:
            continue

    # Группировка записей
    grouped = {drange: [] for drange in config["duration_ranges"]}
    for record in valid_records:
        for drange in config["duration_ranges"]:
            if drange[0] <= record["duration"] < drange[1]:
                grouped[drange].append(record)
                break

    # Подготовка шумов 
    noise_audio = None
    if config["add_noise"] and os.path.exists(config["noise_dir"]):
        noise_files = [f for f in os.listdir(config["noise_dir"]) 
                      if f.endswith(('.wav', '.mp3', '.webm'))]
        if noise_files:
            noise_path = os.path.join(config["noise_dir"], random.choice(noise_files))
            noise_audio = AudioSegment.from_file(noise_path)
            if config["test_mode"]:
                print(f"\nИспользуется шумовой файл: {os.path.basename(noise_path)}")

    # Генерация смесей
    os.makedirs(config["output_dir"], exist_ok=True)
    os.makedirs(config["output_meta_dir"], exist_ok=True)

    successful_mixes = 0
    max_possible_mixes = sum(len(v)*(len(v)-1)//2 for v in grouped.values())
    
    with tqdm(total=max_possible_mixes, desc="Генерация смесей") as pbar:
        for drange in config["duration_ranges"]:
            records = grouped[drange]
            n = len(records)
            
            for i in range(n):
                for j in range(i+1, n):
                    try:
                        rec1, rec2 = records[i], records[j]
                        audio1 = AudioSegment.from_file(rec1["audio_path"])
                        audio2 = AudioSegment.from_file(rec2["audio_path"])
                        
                        # Смешивание голосов
                        pattern = config["patterns"][successful_mixes % len(config["patterns"])]
                        if pattern == "Частичное перекрытие":
                            overlap = min(len(audio1), len(audio2)) // 3
                            mixed = audio1.overlay(audio2, position=len(audio1)-overlap)
                            mixed = audio1[:len(audio1)-overlap] + mixed[-overlap:] + audio2[overlap:]
                        else:
                            mixed = audio1.overlay(audio2)
                        
                        # Наложение зацикленного шума
                        if noise_audio:
                            mixed = add_looped_noise(
                                mixed, 
                                noise_audio, 
                                random.choice(config["snr_levels"])
                            )
                        
                        # Сохранение
                        mix_id = successful_mixes + 1
                        output_name = f"mixed_{mix_id:04d}.wav"
                        output_path = os.path.join(config["output_dir"], output_name)
                        mixed.export(output_path, format="wav")
                        
                        # Метаданные
                        meta = {
                            "id": mix_id,
                            "pattern": pattern,
                            "duration": len(mixed)/1000,
                            "with_noise": noise_audio is not None,
                            "components": [
                                {
                                    "speaker": rec1["speaker_id"],
                                    "duration": rec1["duration"],
                                    "file": os.path.basename(rec1["audio_path"])
                                },
                                {
                                    "speaker": rec2["speaker_id"],
                                    "duration": rec2["duration"],
                                    "file": os.path.basename(rec2["audio_path"])
                                }
                            ]
                        }
                        
                        with open(os.path.join(config["output_meta_dir"], 
                                             output_name.replace('.wav', '.json')), 
                                'w', encoding='utf-8') as f:
                            json.dump(meta, f, indent=2, ensure_ascii=False)
                        
                        successful_mixes += 1
                        pbar.update(1)
                        
                    except Exception as e:
                        if config["test_mode"]:
                            print(f"\nОшибка при создании mix_{successful_mixes+1}: {str(e)}")
                        continue

    print(f"\nГотово! Создано {successful_mixes} смесей")
    if config["test_mode"]:
        print("\nСтатистика:")
        print(f"- Использовано записей: {len(valid_records)}")
        print(f"- Добавлен шум: {'Да' if noise_audio else 'Нет'}")
        print(f"- Папка с результатами: {config['output_dir']}")

def add_looped_noise(audio, noise, snr_db):
    """Добавляет зацикленный шум к аудио"""
    # Рассчитываем нужную громкость шума
    target_noise_rms = audio.rms / (10 ** (snr_db / 20))
    noise_gain_db = 20 * np.log10(target_noise_rms / (noise.rms + 1e-6))
    noise = noise.apply_gain(noise_gain_db)
    
    # Создаем зацикленный шум 
    looped_noise = AudioSegment.empty()
    while len(looped_noise) < len(audio):
        looped_noise += noise
    
    # Обрезаем
    looped_noise = looped_noise[:len(audio)]
    
    # Накладываем шум
    return audio.overlay(looped_noise)

if __name__ == "__main__":
    # Для генерации с шумом:
    generate_mixes(add_noise_flag=True)
    
    # Для генерации без шума:
    # generate_mixes(add_noise_flag=False)


Используется шумовой файл: sample-6.webm


Генерация смесей: 100%|██████████| 7441/7441 [06:03<00:00, 20.49it/s]


Готово! Создано 7441 смесей

Статистика:
- Использовано записей: 183
- Добавлен шум: Да
- Папка с результатами: e:\учеба\пп звук\common_voice_processed\audio\mixed
